# 第 6 周练习 —— 供应链运费预测（微调 Fine Tuning）

## 练习目标

用 **供应链运输合同（ShippingContract）** 数据，走完课程第 6 周的典型流水线：

1. 加载 / 去重 / 探索运费与品类分布
2. 加权抽样得到可训练子集，划分 train / val / test
3. 推到 Hugging Face Hub，再做摘要预处理（Batch）
4. 把对话格式写成 JSONL，微调前沿小模型并评测

## 和本课 Week 6 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 数据策展与分层抽样 | 按运费与品类加权 `np.random.choice` |
| Hub 数据集 | `ShippingContract.push_to_hub` / `from_hub` |
| Chat Fine-Tuning JSONL | `messages` = user 问运费 + assistant 真值 |
| OpenAI Fine-Tuning Job | `gpt-4.1-nano-2025-04-14` + 轮询 `succeeded` |

## 怎么跑

1. 准备好同目录依赖模块：`batch.py`、`contracts.py`、`loaders.py`、`evaluator.py`
2. `.env` 里至少有 `HF_TOKEN`、`OPENAI_API_KEY`
3. 从上到下依次运行；微调格会真实扣费，确认子集规模后再跑


### 导入依赖

把后面要用到的标准库、可视化、Hugging Face、OpenAI 以及本练习自定义模块一次搬进来。


In [ ]:
# ========== 导入：供应链微调流水线所需工具箱 ==========

# os：读环境变量（如 HF_TOKEN）
import os
# Path：注意此处按原代码写成 import Path（可运行性依赖运行环境如何解析该名）
import Path
# numpy：数组运算、归一化权重、加权抽样
import numpy as np
# matplotlib：直方图 / 柱状图 / 饼图 / 散点图
import matplotlib.pyplot as plt
# random：打乱样本顺序（shuffle）
import random
# json：把 messages 序列化进 JSONL
import json

# Batch：对本练习合同做摘要类批处理（create / run / fetch）
from batch import Batch
# Counter：统计各货物品类出现次数
from collections import Counter
# load_dotenv：从 .env 加载密钥
from dotenv import load_dotenv
# login：Hugging Face Hub 登录
from huggingface_hub import login
# ShippingContract：合同领域对象（运费、重量、summary 等）
from contracts import ShippingContract
# ShippingContractLoader：从数据源加载合同列表
from loaders import ShippingContractLoader
# evaluate：用预测函数在 test 集上评测
from evaluator import evaluate
# OpenAI：Files / Fine-Tuning / Chat Completions
from openai import OpenAI
# tenacity：按条件重试（轮询微调任务是否成功）
from tenacity import retry, retry_if_result, wait_fixed

# 加载 .env；override=True 允许覆盖已有环境变量
load_dotenv(override=True)


### 登录 Hugging Face（Hugging Face Hub）

用环境变量里的 `HF_TOKEN` 登录，后续才能 `push_to_hub` / `from_hub`。


In [ ]:
# ========== Hugging Face 登录 ==========

# 从环境读取 HF_TOKEN；缺省会 KeyError，提醒你先配 .env
hf_token: str = os.environ["HF_TOKEN"]
# 登录 Hub；add_to_git_credential=True 便于后续 git/lfs 凭证复用
login(token=hf_token, add_to_git_credential=True)


### 加载合同数据

用自定义 Loader 把运输合同读进内存，后面做探索与抽样。


In [ ]:
# ========== 加载原始合同列表 ==========

# 数据集分割名列表（本练习只用 train；变量保留便于扩展）
dataset_names = ["train"]
# ShippingContractLoader().load()：返回 ShippingContract 对象列表
contracts = ShippingContractLoader().load()


In [ ]:
# ========== 按合同号去重 + 定义探索用绘图函数 ==========

# 用集合记录已见过的 contract_number，列表推导式里顺带 seen.add
seen = set()
contracts: list[ShippingContract] = [
    contract
    for contract in contracts
    # 短路技巧：若编号已在 seen 则丢弃；否则 add 返回 None（假）→ 保留该 contract
    if not (contract.contract_number in seen or seen.add(contract.contract_number))
]

# 打印去重后规模
print(f"There are {len(contracts)} contracts now in the dataset")


def plot_shipping_costs(shipping_costs):
    """绘制运费直方图：看分布是否长尾、均值与最高价。"""
    # 画布尺寸加宽，便于看清长尾
    plt.figure(figsize=(15, 6))
    # 标题里嵌入均值与最高运费（原英文标题字符串保持不变）
    plt.title(
        f"Shipping Costs: Avg {sum(shipping_costs) / len(shipping_costs):,.2f} and highest {max(shipping_costs):,}\n"
    )
    plt.xlabel("Shipping Cost ($)")
    plt.ylabel("Count")
    # bins 按 1000 美元一档，从 0 到 100000
    plt.hist(shipping_costs, rwidth=0.7, color="orange", bins=range(0, 100000, 1000))
    plt.show()


def plot_categories(categories, counts):
    """显示数据集中类别的分布（柱状图 + 柱顶标注数量）。"""
    plt.figure(figsize=(15, 6))

    # 金色柱：每个品类的合同数
    plt.bar(categories, counts, color="goldenrod")
    plt.title("How many in each category")
    plt.xlabel("Categories")
    plt.ylabel("Count")
    # 品类名可能较长：旋转 30° 并右对齐，避免重叠
    plt.xticks(rotation=30, ha="right")

    # 在每根柱子顶部写上千分位格式的计数
    for i, v in enumerate(counts):
        plt.text(i, v, f"{v:,}", ha="center", va="bottom")

    plt.show()


In [ ]:
# ========== 画出「全量合同」的运费分布 ==========

# 抽出每份合同的 total_shipping_cost
shipping_costs = [contract.total_shipping_cost for contract in contracts]

# 调用上一格定义的直方图
plot_shipping_costs(shipping_costs)


In [ ]:
# ========== 画出「全量合同」的货物品类分布 ==========

# Counter：goods_description 枚举值 → 出现次数
category_counts = Counter([contract.goods_description.value for contract in contracts])
# 品类名序列（柱状图 x 轴）
categories = category_counts.keys()
# 与 categories 对齐的计数列表
counts = [category_counts[category] for category in categories]

plot_categories(categories, counts)


In [ ]:
# ========== 找到最便宜 / 最贵合同及其下标 ==========

# 按 total_shipping_cost 取最小 / 最大运费值
min_price = min(contracts, key=lambda x: x.total_shipping_cost).total_shipping_cost
max_price = max(contracts, key=lambda x: x.total_shipping_cost).total_shipping_cost

# 在列表里定位第一个等于该极值的下标（供下一格 print .full）
min_index = next(
    i for i, c in enumerate(contracts) if c.total_shipping_cost == min_price
)
max_index = next(
    i for i, c in enumerate(contracts) if c.total_shipping_cost == max_price
)

# 打印极值与下标，便于人工抽查原文
print(min_price, min_index)
print(max_price, max_index)


In [ ]:
# ========== 查看最便宜合同的完整文本 ==========

# .full：合同全文（字段拼成可读长文本）
print(contracts[min_index].full)


In [ ]:
# ========== 查看最贵合同的完整文本 ==========

print(contracts[max_index].full)


In [ ]:
# ========== 加权抽样：抬高「贵单」权重，压低若干热门品类 ==========

# 固定随机种子，保证可复现
np.random.seed(42)

# 目标子集大小
SIZE = 160

# 运费数组（float）与品类名数组，一一对齐
prices = np.array([contract.total_shipping_cost for contract in contracts], dtype=float)
categories = np.array([contract.goods_description.value for contract in contracts])
# 把运费线性归一化到 [0,1]，贵单基础权重更大
p = (prices - prices.min()) / (prices.max() - prices.min())

# 以归一化运费为初始权重
w = p
# 若干常见品类权重再乘 0.05：降低它们被抽中的概率，避免样本被它们淹没
w[categories == "Electronics"] *= 0.05
w[categories == "Machinery"] *= 0.05
w[categories == "Toys"] *= 0.05
w[categories == "Pharmaceuticals"] *= 0.05

# 归一化成概率分布（和为 1）
w = w / w.sum()
# 无放回加权抽样 SIZE 个下标
idx = np.random.choice(len(contracts), size=SIZE, replace=False, p=w)
# 按下标取出合同对象，得到 sample
sample = [contracts[i] for i in idx]


In [ ]:
# ========== 检查抽样后运费分布是否仍合理 ==========

prices = [contract.total_shipping_cost for contract in sample]
plot_shipping_costs(prices)


In [ ]:
# ========== 打乱 sample 顺序（避免按抽样下标残留顺序偏差） ==========

random.seed(42)
random.shuffle(sample)

# 打乱后再看一次运费直方图（分布应与上一格类似，只是顺序变了）
prices = [contract.total_shipping_cost for contract in sample]
plot_shipping_costs(prices)


In [ ]:
# ========== 抽样后的品类柱状图 ==========

category_counts = Counter([contract.goods_description.value for contract in sample])
categories = category_counts.keys()
counts = [category_counts[category] for category in categories]

plot_categories(categories, counts)


In [ ]:
# ========== 品类占比圆环图（Pie / Donut） ==========

def plot_categories_pie(categories, counts):
    plt.figure(figsize=(12, 10))

    # 饼图：autopct 显示整数百分比；startangle=90 让第一块从正上方开始
    plt.pie(counts, labels=categories, autopct="%1.0f%%", startangle=90)

    # 在中心叠加白圆 → 视觉上变成圆环图（Donut）
    centre_circle = plt.Circle((0, 0), 0.70, fc="white")
    fig = plt.gcf()
    fig.gca().add_artist(centre_circle)
    plt.title("Categories")

    # 相等长宽比，保证饼图是正圆而不是椭圆
    plt.axis("equal")

    plt.show()


# 用上一格的 categories / counts 绘图
plot_categories_pie(categories, counts)


In [ ]:
# ========== 探索：重量 vs 运费是否简单线性相关 ==========

def plot_price_vs_weight_corelation(sample):
    # 运费如何随重量变化？

    # x：总重量（kg）；y：总运费（USD）
    weights = [contract.total_weight for contract in sample]
    prices = [contract.total_shipping_cost for contract in sample]

    # 散点很密时用很小的点尺寸 s=0.2
    plt.figure(figsize=(15, 8))
    plt.scatter(weights, prices, s=0.2, color="red")

    # 轴标签与标题（英文保持原样，属于图上展示文案）
    plt.xlabel("Weight in kgs")
    plt.ylabel("Price in USD")
    plt.title("Is there a simple correlation with weight?")

    plt.show()


plot_price_vs_weight_corelation(sample)


### 推送到 Hugging Face Hub

把抽样后的 train / val / test 上传为 Hub 数据集，方便以后 `from_hub` 复现实验。


In [ ]:
# ========== 划分 train/val/test 并 push_to_hub ==========

# Hub 用户名（仓库前缀）；改成你自己的账号才能成功推送
username = "gathondu"
# 完整数据集名：用户名/仓库名
full = f"{username}/supply-chain-contracts-full"

# 固定切片：128 训练 / 16 验证 / 其余测试（合计应 = len(sample)=160）
train: list[ShippingContract] = sample[:128]
val: list[ShippingContract] = sample[128:144]
test: list[ShippingContract] = sample[144:]

# 推送到 Hub（需已 login 且对 username 有写权限）
ShippingContract.push_to_hub(full, train, val, test)


## 数据预处理

### 从 Hugging Face 拉取 → Batch 生成摘要 → 再推回 Hub

这一步把「全文合同」变成更适合喂给模型的 `summary`，并保存为预处理后的数据集。


In [ ]:
# ========== 拉取全量切分 → Batch 摘要 → 再划分并推送预处理版 ==========

# 与上一格 push 的仓库名一致
dataset = f"{username}/supply-chain-contracts-full"

# 从 Hub 拉回三个列表
train, val, test = ShippingContract.from_hub(dataset)

# 合并，方便统一跑 Batch
contracts = train + val + test

# Batch 三步：创建任务 → 执行 → 取回结果写回 contract.summary
Batch.create(contracts)
Batch.run()
Batch.fetch()
# 抽查第 110 条摘要是否像「短文本特征」
print(contracts[110].summary)
# 按原切分边界重新切开（顺序与合并前一致的前提：from_hub 顺序稳定）
train: list[ShippingContract] = contracts[:128]
val: list[ShippingContract] = contracts[128:144]
test: list[ShippingContract] = contracts[144:]

# 预处理后的新仓库名
preprocessed = f"{username}/supply-chain-contracts-preprocessed"

# 推送带 summary 的版本，供微调格使用
ShippingContract.push_to_hub(preprocessed, train, val, test)


## 微调前沿模型（`gpt-4.1-nano-2025-04-14`）

把预处理后的合同写成 Chat JSONL，上传后创建 Fine-Tuning Job，轮询完成再用 test 集评测。


In [ ]:
# ========== JSONL → 上传 → 创建微调任务 → 轮询 → 评测 ==========

# 使用预处理后的 Hub 数据集
dataset = f"{username}/supply-chain-contracts-preprocessed"

train, val, test = ShippingContract.from_hub(dataset)
print(
    f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items"
)

# 默认读取 OPENAI_API_KEY
openai = OpenAI()


def messages_for(contract):
    """训练用：user 提问 + assistant 给出真值运费（监督信号）。"""
    # prompt 字符串保持英文原样（影响模型行为）
    message = f"Estimate the shipping cost for this contract. Respond with the shipping cost, no explanation\n\n{contract.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${contract.total_shipping_cost} USD"},
    ]


def make_jsonl(contracts):
    """手写拼 JSONL：每行 {"messages": [...]}（注意原实现用字符串拼接）。"""
    result = ""
    for contract in contracts:
        messages = messages_for(contract)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str + "}\n"
    return result.strip()


def write_jsonl(contracts, filename):
    """把 JSONL 文本写到指定文件路径。"""
    with open(filename, "w") as f:
        jsonl = make_jsonl(contracts)
        f.write(jsonl)


# 确保本地 jsonl/ 目录存在
Path("jsonl").mkdir(parents=True, exist_ok=True)
# 写出训练 / 验证文件
write_jsonl(train, "jsonl/fine_tune_train.jsonl")
write_jsonl(val, "jsonl/fine_tune_validation.jsonl")

# 上传训练文件（purpose 必须是 fine-tune）
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

# 上传验证文件
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

# 创建微调任务：基座模型、1 epoch、batch_size=1、后缀 supply-chain
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    hyperparameters={
        "n_epochs": 1,
        "batch_size": 1,
    },
    suffix="supply-chain",
)

# 列出最近 1 个 job，取其 id（约定：刚创建的就是最新）
first_job = next(iter(openai.fine_tuning.jobs.list(limit=1)))
job_id = first_job.id

# 先 retrieve 一次，拿到 job 对象（后面取 fine_tuned_model 会用到这个引用）
job = openai.fine_tuning.jobs.retrieve(job_id)


# tenacity：若返回值为假（未成功）则每 10 秒重试
@retry(retry=retry_if_result(lambda result: not result), wait=wait_fixed(10))
def wait_for_job(job_id):
    job = openai.fine_tuning.jobs.retrieve(job_id)
    print(job.status)
    # True 才停止重试
    return job.status == "succeeded"


# 阻塞直到 succeeded
wait_for_job(job_id)
# 注意：这里用的是循环外层的 job 对象字段；以原逻辑为准不改写
fine_tuned_model_name = job.fine_tuned_model
print(fine_tuned_model_name)


def test_messages_for(contract):
    """推理用：只给 user，不给 assistant 真值。"""
    message = f"Estimate the shipping cost for this contract. Respond with the shipping cost, no explanation\n\n{contract.summary}"
    return [
        {"role": "user", "content": message},
    ]


def gpt_4__1_nano_fine_tuned(contract):
    """用微调模型预测单条合同运费文本。"""
    response = openai.chat.completions.create(
        model=fine_tuned_model_name, messages=test_messages_for(contract), max_tokens=7
    )
    return response.choices[0].message.content


# 在 test 集上跑课程 evaluate
evaluate(gpt_4__1_nano_fine_tuned, test)
